# 🤖 Social Media Post AI Agent
### Build Story: From Architecture to Working Demo

---

## 🎯 Objective

Build an AI agent that creates consistent, on-brand social media posts for any company or product. The agent remembers past work — so on return visits, the user skips straight to generating new posts in the same style without re-researching anything.

**The problem this solves:** Creating social media content manually is time-consuming, inconsistent across sessions, and hard to scale. A simple ChatGPT prompt isn't enough because it has no memory of past brand guidelines, no ability to search for company info, and no way to run multiple platform-specific agents simultaneously.

---

## 🧠 Agentic Pattern: Sequential + Parallel Workflow

This project combines two patterns from class:

1. **Sequential Workflow** (like `orchestrator.py` from class) — Research → Style Guide → Image Selection → Post Generation → Scheduling. Each stage depends on the previous one.

2. **Parallel Multi-Agent** (like `07_workflow_multitasking.ipynb` from class) — Within post generation, one specialized agent per platform runs simultaneously using `ThreadPoolExecutor`. An Instagram Specialist, Twitter Specialist, and Blog Post Writer all work at the same time.

3. **ReAct + Function Calling** (upgraded from `02_react_agent.ipynb` from class) — The research agent reasons through Thought/Action/Observation steps and calls tools (web search, URL fetch, document read, ask user) via OpenAI function calling instead of regex parsing.

---

## 📁 Folder Structure Created Automatically

```
AI Storage/
  {Company}/
    {Company} Company Report.json
    Products/{Product} Product Report.json
    Style Guides/{Product} Style Guide.json
    Images/{Product}/          ← grows over time
    Created Posts/{date} – Request {Product}/
      Post 1/caption.txt + post_image.png
    Log Files/
```

---

**Setup:** Fill in `.env` with your `OPENAI_API_KEY` before running. See `README.md` for full setup instructions.

---
## 📦 Step 0: Imports & Config

Imports all local agent modules. Each module is a separate Python file in the same folder — this keeps the code organized and mirrors the class project structure where each agent lives in its own file (e.g. `twitter_agent.py`, `parallel_workflow.py`).

In [ ]:
import os, json, re, datetime, time
from pathlib import Path

from agent_storage  import Storage
from agent_research import ResearchAgent
from agent_posts    import PLATFORM_AGENTS, build_context, parse_platforms
from agent_parallel import run_all_posts
from agent_schedule import ScheduleAgent
from agent_logger   import Logger
from agent_utils    import confirm, print_receipt

print('✅ Imports OK')

---
## 🔑 Step 1: API Keys

Uses `python-dotenv` to load keys from a `.env` file — the same pattern as the class `openai_client.py` which calls `load_dotenv()` at the top of the file. Keys are never hardcoded in the notebook itself, which is important for the GitHub repo (`.env` is in `.gitignore`).

`OPENAI_REVIEW_KEY` uses the same key but feeds into the cheaper `gpt-3.5-turbo` model used by the A/B reviewer — keeping costs down while still scoring post quality.

In [ ]:
from dotenv import load_dotenv

# Loads all keys from .env automatically — same pattern as class openai_client.py
load_dotenv()

OPENAI_API_KEY        = os.getenv('OPENAI_API_KEY')
OPENAI_REVIEW_KEY     = os.getenv('OPENAI_API_KEY')   # same key, cheaper model used internally
GOOGLE_CALENDAR_ID    = os.getenv('GCAL_ID', 'primary')
GCAL_CREDENTIALS_JSON = os.getenv('GCAL_CREDENTIALS', 'credentials.json')

if not OPENAI_API_KEY:
    raise ValueError('OPENAI_API_KEY not found. Did you fill in your .env file?')

print('✅ Keys loaded from .env')

---
## 📁 Step 2: Storage Paths

`Storage` is the file system manager for the whole project. It handles all folder creation, JSON reading/writing, and image management. Centralizing file I/O in one class means no agent ever writes files directly — they all go through `Storage`. This makes it easy to change the folder structure in one place without touching any agent code.

In [ ]:
PROJECT_ROOT = Path('.')   # same folder as this notebook
storage = Storage(PROJECT_ROOT / 'AI Storage')
print(f'📂 AI Storage: {storage.root.resolve()}')

---
## 🗣️ Step 3: Parse User Request

**Prompt Engineering — Request Parsing**

The user types a natural language request and `parse_request()` extracts structured fields from it using regex patterns. This is the first prompt engineering decision in the project: rather than asking the user to fill out a form, they can write naturally and the system figures out the intent.

The parsed output is called a **receipt** — it's the order that drives the rest of the workflow.

**Image mode parsing** is also handled here. Platform defaults are defined in `IMAGE_PLATFORM_DEFAULTS` in `agent_utils.py` — easy to find and modify:
- Instagram → `Provided Images` by default
- Twitter / Blog → `No` by default
- `"with image"` on any platform → `Provided Images`
- `"AI generated"` explicitly → `AI Generated`
- `"without image"` → `No`

**Try changing `USER_REQUEST` below** to see how the parser handles different inputs.

In [ ]:
# ── Change this to your request ────────────────────────────────────────────
USER_REQUEST = "Create 3 Instagram posts for Rita's Kiwi Melon in June + Schedule. Make it interactive please."

from agent_utils import parse_request
receipt = parse_request(USER_REQUEST)
print_receipt(receipt)

---
## 🔍 Step 4: Company & Product Lookup

**Agentic Pattern: ReAct + Function Calling**

This is where the research agent runs. It uses a combination of two patterns from class:

**From `02_react_agent.ipynb`:** The ReAct prompt (`REACT_PROMPT` in `agent_research.py`) tells GPT to reason step by step — Thought → Action → PAUSE → Observation → Answer. You can watch the thinking happen live in the output below.

**Upgraded from class:** The class version used regex (`ACTION_RE`) to parse action names out of raw GPT text. This project upgrades to **OpenAI function calling** — GPT signals which tool it wants via a structured `tool_calls` object instead of plain text. This is more reliable because GPT can't accidentally misspell the action name.

**Four tools available to the agent:**
| Tool | When used |
|---|---|
| `web_search` | Company/product has an online presence |
| `fetch_url` | User provided a specific URL |
| `read_document` | User uploaded a file (PDF, txt) |
| `ask` | Nothing found — agent pauses and asks the user |

**On return visits:** If the company and product reports already exist locally, the agent loads them instantly and skips all research — the user just hits Enter.

In [ ]:
company_report, product_report, resolved_company, resolved_product = researcher.resolve(
    company_name=receipt['company'],
    product_name=receipt['product'],
    provided_url=provided_url,
    uploaded_file=uploaded_file,
)

# Update receipt with the canonical names resolved by the agent
# (e.g. 'Rita\'s' becomes 'Rita\'s Water Ice' after fuzzy match / official name lookup)
receipt['company'] = resolved_company
receipt['product'] = resolved_product
print(f'  ✅ Receipt updated: company=[{resolved_company}], product=[{resolved_product}]')

---
## 📖 Step 5: Review Reports (Optional)

Gives the user a chance to inspect the generated Company and Product reports before continuing. Since these are saved as JSON files in `AI Storage/`, the user can also edit them directly in any text editor and press Enter to continue with the updated version.

**On return visits:** This step is quick — the reports are already accurate from the first run, so the user typically just hits Enter to skip.

In [ ]:
want_review = confirm('Would you like to review the Company or Product report before continuing? (y/N)')
if want_review:
    choice = input("Type 'company', 'product', or 'both': ").strip().lower()
    if choice in ('company', 'both'):
        print(json.dumps(company_report, indent=2))
    if choice in ('product', 'both'):
        print(json.dumps(product_report, indent=2))
    input('\nPress Enter when done reviewing...')

---
## 🎨 Step 6: Style Guide

**Prompt Engineering — Style Consistency**

The style guide is the key to consistent posts across sessions. It captures the brand vibe, tone, color palette, emoji usage, and visual themes — without storing any specific product facts. That separation is intentional: the style guide is about *how* the brand communicates, not *what* it sells.

**Reference posts:** The user can add URLs to real social media posts they like. The agent stores these as references in the style guide and uses them for inspiration on future runs.

**On return visits:** If a style guide already exists for this product, the agent loads it and only asks if the user wants to update the reference posts — nothing else changes unless requested.

In [ ]:
style_guide = researcher.resolve_style_guide(
    company_report=company_report,
    product_report=product_report,
)

---
## ✅ Step 7: Confirm Receipt

Shows the parsed receipt and lets the user modify any field before generation begins. This is the last chance to make changes before the agents start working.

**How to modify:** Type `modify <field> to <value>` — for example `modify num_posts to 5` or `modify images to AI Generated`. Press Enter or type `all good` to confirm.

**On return visits:** The receipt is re-parsed from the new request each time, so the user can change platforms, post count, or any other field without touching any files.

In [ ]:
from agent_utils import interactive_receipt_editor
receipt = interactive_receipt_editor(receipt)
print_receipt(receipt)

---
## 🖼️ Step 7.5: Image Selection

**Image Feature — Three Modes**

This step only runs if `images` in the receipt is not `No`. It manages the per-product image library stored in `AI Storage/{Company}/Images/{Product}/`.

| Mode | What happens |
|---|---|
| `Provided Images` | Shows library, user selects images, then chooses: enhance real photo or use as inspiration |
| `AI Generated` | Optionally shows library for reference, then DALL-E generates fresh |
| `No` | Skips this step entirely |

**Fallback:** If `Provided Images` is selected but no images exist in the library and the user hits Enter without adding any, the mode automatically falls back to `AI Generated`.

**Guardrail:** Only `.png` and `.jpg/.jpeg` files are accepted. Any other format is rejected with a clear message.

**Caption-first design:** Image generation happens *after* the caption is written in Step 8. The caption is passed as context to DALL-E or the image edit endpoint so the image and caption feel cohesive — same mood, same message. This was a deliberate prompt engineering decision (see `_build_image_prompt()` in `agent_base.py`).

In [ ]:
from agent_storage import ALLOWED_IMAGE_EXTENSIONS

image_mode       = receipt.get('images', 'No')
selected_images  = []   # product photos for post image generation
reference_images = []   # style reference screenshots for GPT Vision
style_vibe       = style_guide.get('vibe', '')

if image_mode == 'No':
    print('⏭️  Image mode is No — skipping image selection.')
    # Still offer style references even when no post image needed
    refs = storage.list_references(receipt['company'], receipt['product'])
    if refs:
        print(f'\n🖼️  Found {len(refs)} style reference(s) — these will guide post style.')
        for i, r in enumerate(refs, 1):
            print(f'    {i}. {r.name}')
        reference_images = refs

else:
    company = receipt['company']
    product = receipt['product']

    img_dir = storage.images_dir(company, product)
    img_dir.mkdir(parents=True, exist_ok=True)
    existing = storage.list_images(company, product)

    print(f'\n🖼️  Image library for [{product}]: {img_dir}')
    if existing:
        print(f'  Found {len(existing)} product photo(s):')
        for i, p in enumerate(existing, 1):
            print(f'    {i}. {p.name}')
    else:
        print('  No product photos in library yet.')

    print()
    print('  Product photo commands (used for actual post image):')
    print('    add <filepath>     — copy a product photo into the library')
    print('    select <number>    — select a photo for this post')
    print('    remove <number>    — remove a photo from the library')
    print('    Enter              — done')
    print()

    while True:
        cmd = input('  > ').strip()
        if not cmd:
            break
        parts  = cmd.split(None, 1)
        action = parts[0].lower()
        arg    = parts[1].strip() if len(parts) > 1 else ''
        if action == 'add' and arg:
            added = storage.add_image_to_library(company, product, arg)
            if added:
                existing = storage.list_images(company, product)
        elif action == 'select' and arg.isdigit():
            idx = int(arg) - 1
            if 0 <= idx < len(existing):
                chosen = existing[idx]
                if chosen not in selected_images:
                    selected_images.append(chosen)
                    print(f'  ✅ Selected: {chosen.name}')
            else:
                print(f'  ❌ Invalid number.')
        elif action == 'remove' and arg.isdigit():
            idx = int(arg) - 1
            if 0 <= idx < len(existing):
                target = existing[idx]
                target.unlink()
                existing = storage.list_images(company, product)
                selected_images = [p for p in selected_images if p != target]
                print(f'  🗑️  Removed: {target.name}')
        else:
            print('  add <filepath> | select <number> | remove <number> | Enter to continue')

    if image_mode == 'Provided Images' and not selected_images:
        if not existing:
            print('\n  ℹ️  No product photos found — falling back to AI Generated.')
            image_mode = 'AI Generated'
            receipt['images'] = 'AI Generated'

# ── Style references (separate from product photos) ───────────────────────
# Screenshots of posts to imitate — passed to GPT Vision during drafting
# using the class HumanMessage image_url pattern from 01_langchain_basics.ipynb
ref_dir = storage.references_dir(receipt['company'], receipt['product'])
ref_dir.mkdir(parents=True, exist_ok=True)
existing_refs = storage.list_references(receipt['company'], receipt['product'])

print(f'\n🎨 Style references for [{receipt["product"]}]: {ref_dir}')
if existing_refs:
    print(f'  Found {len(existing_refs)} reference(s):')
    for i, r in enumerate(existing_refs, 1):
        print(f'    {i}. {r.name}')
else:
    print('  No style references yet.')

print()
print('  Style reference commands (screenshots of posts to imitate):')
print('    addref <filepath>  — add a post screenshot as a style reference')
print('    rmref <number>     — remove a style reference')
print('    Enter              — done, use all existing references')
print()

while True:
    cmd = input('  ref> ').strip()
    if not cmd:
        break
    parts  = cmd.split(None, 1)
    action = parts[0].lower()
    arg    = parts[1].strip() if len(parts) > 1 else ''
    if action == 'addref' and arg:
        added = storage.add_reference_to_library(receipt['company'], receipt['product'], arg)
        if added:
            existing_refs = storage.list_references(receipt['company'], receipt['product'])
    elif action == 'rmref' and arg.isdigit():
        idx = int(arg) - 1
        if 0 <= idx < len(existing_refs):
            existing_refs[idx].unlink()
            existing_refs = storage.list_references(receipt['company'], receipt['product'])
            print(f'  🗑️  Removed.')
    else:
        print('  addref <filepath> | rmref <number> | Enter to continue')

reference_images = storage.list_references(receipt['company'], receipt['product'])
print(f'\n✅ Image mode: {image_mode}')
print(f'   Product photos selected : {[p.name for p in selected_images] or "none"}')
print(f'   Style references loaded : {[r.name for r in reference_images] or "none"}')

---
## ✍️ Step 8: Generate Posts (Parallel Workflow)

**Agentic Pattern: Parallel Multi-Agent Execution**

This step directly mirrors `07_workflow_multitasking.ipynb` from class. The same three components:

1. **Specialized agents** — `InstagramAgent`, `TwitterAgent`, `BlogAgent` each inherit from `BaseAgent` (same as `TwitterAgent` inheriting from `BaseAgent` in class). Each has one `execute()` method.

2. **`ParallelWorkflow`** — uses `ThreadPoolExecutor` to run all agents simultaneously (identical to class `ParallelWorkflow`). Multiple posts across multiple platforms all generate at the same time.

3. **A/B scoring loop** — each agent drafts a post, a cheaper reviewer model scores it (0-10), and the loop keeps the best version up to `max_tries` attempts. This is in `_ab_loop()` in `agent_base.py`.

**Prompt Engineering — Caption + Image Cohesion:**
Within each agent's `execute()`, the caption is generated first via the A/B loop. The final caption is then passed as context to the image generation step — so DALL-E or the image edit endpoint knows what story the post is telling. This was a deliberate design choice: image_context alone produces generic images, but caption + image_context produces images that match the post's message.

**New vs class:** `base_brief` now carries `image_mode`, `selected_images`, and `style_vibe` alongside context — these flow through to every agent automatically via the `{**base_brief, 'post_num': n}` spread.

In [ ]:
# ============================================================================
# STEP 8: PARALLEL POST GENERATION
# ============================================================================
# Mirrors the class notebook (07_workflow_multitasking.ipynb) pattern:
#   1. Build a shared context (replaces product_brief from class)
#   2. Instantiate one specialized agent per platform
#   3. Run all agents in parallel using ParallelWorkflow + ThreadPoolExecutor
#   4. Display results in the same bordered format as class output
#
# New vs class: base_brief now carries image_mode, selected_images, style_vibe
# so each agent knows what to do with images after generating its caption.
# ============================================================================

# ── 1. Build shared context from reports ───────────────────────────────
context = build_context(
    company=company_report,
    product=product_report,
    style=style_guide,
    extra=receipt.get('additional_info', ''),
    month=receipt.get('when', ''),
)

# ── 2. Build agent list — one per platform, matching class pattern ─────────
platforms = parse_platforms(receipt.get('platforms', 'instagram'))
n_posts   = int(receipt.get('num_posts', 1))

agents_per_post = [
    [
        PLATFORM_AGENTS[p](openai_key=OPENAI_API_KEY, review_key=OPENAI_REVIEW_KEY)
        for p in platforms
        if p in PLATFORM_AGENTS
    ]
    for _ in range(n_posts)
]

print(f'🚀 Starting post generation...\n' + '=' * 70)
print(f'📋 Platforms : {", ".join(p.capitalize() for p in platforms)}')
print(f'   Posts     : {n_posts} per platform')
print(f'   Images    : {image_mode}')
print('=' * 70 + '\n')

print('✅ Initialized agents:')
for agent in agents_per_post[0]:
    print(f'   • {agent.name}')
print()

# ── 3. Run parallel workflow — image fields flow through automatically ─────
base_brief = {
    'context':         context,
    'total_posts':     n_posts,
    'image_mode':      image_mode,
    'reference_images': reference_images,
    'selected_images': selected_images,
    'style_vibe':      style_vibe,
}

start_time = time.time()
posts      = run_all_posts(agents_per_post, base_brief)
elapsed    = time.time() - start_time

print(f'\n✨ All agents finished in {elapsed:.2f} seconds')
print('=' * 70)

# ── 4. Display results — same bordered format as class notebook ───────────
print('\n📊 GENERATED SOCIAL MEDIA CONTENT')
print('=' * 70 + '\n')

for i, post in enumerate(posts, 1):
    has_image = post.get('image_bytes') is not None
    label     = f'Post {i} [{post.get("platform","?").upper()}] — Score: {post.get("score", "N/A")} {"🖼️" if has_image else ""}'
    print(f'┌─ {label} ' + '─' * max(0, 68 - len(label)))
    print('│')
    for line in post.get('caption', '').split('\n'):
        print(f'│ {line}')
    print('└' + '─' * 69)
    print()

print('=' * 70)
print(f'✅ Generation complete! Total time: {elapsed:.2f}s')
print('=' * 70)

---
## 💾 Step 9: Save Posts & Images

`save_posts()` creates the output folder structure and writes `caption.txt` for each post. Images are saved separately via `save_image()` which writes the raw PNG bytes returned by DALL-E or the image edit endpoint.

The receipt is also saved as `Receipt of Creation.txt` alongside the posts so there's always a record of what was requested.

In [ ]:
# Save captions and receipt to the output folder
output_dir = storage.save_posts(
    company=receipt['company'],
    product=receipt['product'],
    posts=posts,
    receipt=receipt,
)

# Save any generated or enhanced images into their post subfolder
# image_bytes is None if image_mode was No or generation failed
for i, post in enumerate(posts, 1):
    if post.get('image_bytes'):
        post_dir = output_dir / f'Post {i}'
        storage.save_image(post_dir, post['image_bytes'], filename='post_image.png')

print(f'\n📁 All posts saved to: {output_dir}')

---
## 📅 Step 10: Schedule Posts (Optional)

If `schedule` in the receipt is `Yes`, this step:
1. Asks GPT to suggest ideal posting dates for the given month and platform
2. Shows the suggestions and asks the user to confirm (up to 3 attempts)
3. Pushes confirmed dates to Google Calendar via the Google Calendar API

**Setup required:** Google Calendar credentials JSON from Google Cloud Console. See `README.md` for step-by-step instructions. Skipped automatically if `schedule` is `No`.

In [ ]:
if receipt.get('schedule') == 'Yes':
    scheduler = ScheduleAgent(
        openai_key=OPENAI_API_KEY,
        credentials_json=GCAL_CREDENTIALS_JSON,
        calendar_id=GOOGLE_CALENDAR_ID,
    )
    scheduler.run(
        receipt=receipt,
        posts=posts,
        output_dir=output_dir,
    )
else:
    print('⏭️  Scheduling skipped.')

---
## 📝 Step 11: Write Log

Writes a structured audit log to `AI Storage/{Company}/Log Files/`. The log records the full receipt, a summary of the company and product reports, the style guide vibe, every generated caption with its A/B score, and the output folder path.

This mirrors the audit trail pattern from `agency_core.py` in class — every run is traceable.

In [ ]:
logger   = Logger(storage)
log_path = logger.write(
    receipt=receipt,
    company_report=company_report,
    product_report=product_report,
    style_guide=style_guide,
    posts=posts,
    output_dir=output_dir,
)
print(f'\n🎉 All done! Log: {log_path}')

---
## 🗂️ File Manager — View & Update Core Files

In addition to creating posts, the agent can view and update the three core files:
**Company Report**, **Product Report**, and **Style Guide**.
Logs are view-only.

**How intent detection works:**

Before every message, a small GPT call runs `INTENT_PROMPT` to classify what the user wants:
- `create_posts` → normal post creation flow
- `manage` → file manager side flow
- `quit` → exit

This is the same classification pattern as `sentiment_analysis` from `prompts_text_usecases.py`
in class — natural language in, structured label out.
Extended here to also extract company, product, and file_type so the agent knows exactly what to act on.

**Guardrail:** Only `company_report`, `product_report`, and `style_guide` can be edited.
Any attempt to edit a log is blocked and redirected to view-only.

**Four options when managing a file:**
1. **Auto-update** — re-runs the full research loop and rewrites the file
2. **Specific change** — user describes change in plain English, GPT applies it, loops until confirmed
3. **View only** — no changes made
4. **Exit** — go back to main loop

The cell below shows an example of loading and displaying a file directly from the notebook:

In [ ]:
# Example: view a company report directly in the notebook
# Change 'company' and 'product' to match your AI Storage folder
company = "Rita's Water Ice"
product = "Kiwi Melon"

if storage.company_exists(company):
    report = storage.load_company_report(company)
    print(f"Company: {report.get('company_name')}")
    print(f"Industry: {report.get('industry')}")
    print(f"Brand voice: {report.get('brand_voice')}")
    print()
    print(json.dumps(report, indent=2))
else:
    print(f"No company report found for [{company}]. Run Step 4 first.")

---
## 🚀 Live Demo

The cells above document how the agent was built step by step.

**To run the full interactive agent** (post creation + file management):

```bash
python demo.py
```

The demo uses three additional class patterns:
- **`simpleChatBot_chatgpt.py`** — full conversation history maintained
- **`generate_text_stream()` from `openai_client.py`** — streaming token-by-token responses
- **`prompts_text_usecases.py` sentiment_analysis pattern** — `INTENT_PROMPT` classifies each message as `create_posts`, `manage`, or `quit` before routing

Or run the cell below to launch directly from this notebook.

In [ ]:
# Launches the interactive terminal UI
# Type naturally: create posts, view/update files, or 'done' to exit
import subprocess
subprocess.run(["python", "demo.py"])